# LLM Serving Benchmark — Analysis

Comparative analysis of TensorRT-LLM, vLLM, and SGLang across workload patterns.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", font_scale=1.1)
PALETTE = {"vllm": "#1f77b4", "trtllm": "#ff7f0e", "sglang": "#2ca02c"}
results_dir = Path("../results")

## Load Results

In [ ]:
combined_path = results_dir / "analysis" / "combined_results.parquet"
if combined_path.exists():
    df = pd.read_parquet(combined_path)
    print(f"Loaded {len(df)} request records")
    print(f"Engines: {df['engine'].unique()}")
    print(f"Workloads: {df['workload'].unique()}")
    df.head()
else:
    print("No results found. Run benchmarks first:")
    print("  bench run")
    print("  bench analyze")

## Time to First Token (TTFT) Analysis

TTFT measures prefill latency — how long until the model starts generating.
This is dominated by prompt processing and KV cache allocation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for engine in df["engine"].unique():
    subset = df[df["engine"] == engine]
    means = subset.groupby("concurrency")["ttft"].mean()
    stds = subset.groupby("concurrency")["ttft"].std()
    axes[0].errorbar(means.index, means.values, yerr=stds.values,
                     label=engine, marker="o", capsize=4, color=PALETTE.get(engine))
axes[0].set_xlabel("Concurrency")
axes[0].set_ylabel("TTFT (seconds)")
axes[0].set_title("TTFT vs Concurrency")
axes[0].legend()
sns.boxplot(data=df, x="engine", y="ttft", hue="engine",
            palette=PALETTE, ax=axes[1], legend=False)
axes[1].set_title("TTFT Distribution")
plt.tight_layout()
plt.show()

## Inter-Token Latency (ITL) Analysis

ITL measures decode latency per token. Spikes indicate memory pressure,
cache misses, or scheduling delays. This is where prefix caching
inefficiencies and memory fragmentation surface.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(data=df, x="engine", y="itl_mean", hue="engine",
               palette=PALETTE, ax=ax, legend=False)
ax.set_xlabel("Engine")
ax.set_ylabel("Mean ITL (seconds)")
ax.set_title("Inter-Token Latency Distribution by Engine")
plt.show()

## Throughput Comparison

Output tokens per second across concurrency levels.

In [ ]:
if "throughput_tps" in df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    means = df.groupby(["engine", "concurrency"])["throughput_tps"].mean().reset_index()
    sns.barplot(data=means, x="concurrency", y="throughput_tps", hue="engine",
                palette=PALETTE, ax=ax)
    ax.set_xlabel("Concurrency")
    ax.set_ylabel("Throughput (tokens/sec)")
    ax.set_title("Throughput vs Concurrency")
    plt.show()

## Latency Heatmap

End-to-end latency across the full configuration matrix.

In [ ]:
pivot = df.groupby(["engine", "concurrency"])["e2e_latency"].mean().unstack()
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax)
ax.set_title("E2E Latency Heatmap (seconds)")
plt.show()

## Statistical Summary

In [ ]:
summary = df.groupby(["engine", "workload"]).agg(
    ttft_mean=("ttft", "mean"),
    ttft_p95=("ttft", lambda x: np.percentile(x, 95)),
    itl_mean=("itl_mean", "mean"),
    e2e_mean=("e2e_latency", "mean"),
    e2e_p99=("e2e_latency", lambda x: np.percentile(x, 99)),
    count=("ttft", "count"),
).round(4)
summary